# Week 4: Working with textual data — answers

One possible set of answers to the [assignment](pre-processing.ipynb).
Other code is fine if it gives the same results.

Parts 1–4 are the main assignment. Stemming and lemmatisation are optional.

## Before you start

Part 4 uses pandas and scikit-learn. If an import in that part fails, remove the
`#` from the line below and run it once.

In [1]:
# %pip install pandas scikit-learn

## The texts

Run this cell. These six made-up news texts are our practice data.

In [2]:
texts = [
    "  <b>Climate policy</b> divides the city council!  ",
    "The council debates climate policy.",
    "Climate protests grow as students call for climate action.",
    "A new tram line opens in the city.",
    "City council: the new transport plan is not very good.",
    "Readers discuss climategate in online comments.",
]

print(len(texts))
print(texts[0])

6
  <b>Climate policy</b> divides the city council!  


## 1. Clean the texts

**First, run the example below.** It removes spaces from the ends of the first
text, converts it to lowercase, and removes the two HTML tags.

In [3]:
text = texts[0]
text = text.strip()
text = text.lower()
text = text.replace("<b>", "")
text = text.replace("</b>", "")

print(text)

climate policy divides the city council!


**Now write a loop** that applies the same steps to every text in `texts`.
Add each result to `cleaned_texts` and print the list.
Keep the punctuation for now.

Check: the first cleaned text should be
`"climate policy divides the city council!"`.

In [4]:
cleaned_texts = []

for text in texts:
    text = text.strip()
    text = text.lower()
    text = text.replace("<b>", "")
    text = text.replace("</b>", "")
    cleaned_texts.append(text)

print(cleaned_texts)

['climate policy divides the city council!', 'the council debates climate policy.', 'climate protests grow as students call for climate action.', 'a new tram line opens in the city.', 'city council: the new transport plan is not very good.', 'readers discuss climategate in online comments.']


## 2. Count mentions with a regular expression

Use the original list `texts` for this part. **Loop through the texts and print
how often the word “climate” occurs in each one.**

Use `re.findall()`, `len()`, and `re.IGNORECASE`, as in the slides. The pattern
`r"\bclimate\b"` matches “climate” as a whole word.

In [5]:
import re

for text in texts:
    matches = re.findall(r"\bclimate\b", text, re.IGNORECASE)
    print(len(matches))

1
1
2
0
0
0


**In the text cell below**, explain why “climategate” in the last text is not
counted. Do these word counts tell you whether a text supports climate policy?

“Climategate” is one longer word, so it does not match the whole-word pattern
for “climate”. The counts are 1, 1, 2, 0, 0, and 0.

A mention does not tell us whether the text supports climate policy. It could,
for example, report disagreement or criticism.

## 3. Tokenisation and stopwords

**Run the next cell.** It lowercases the fifth text and extracts tokens of at
least two characters. Punctuation separates the tokens.

In [6]:
import re

text = texts[4]
tokens = re.findall(r"\w{2,}", text.lower())
print(tokens)

['city', 'council', 'the', 'new', 'transport', 'plan', 'is', 'not', 'very', 'good']


**Write a loop** that keeps only the tokens that are **not** in `stop_words`.
Add them to `filtered_tokens` and print that list.

In [7]:
stop_words = ["the", "is", "not", "very"]
filtered_tokens = []

for word in tokens:
    if word not in stop_words:
        filtered_tokens.append(word)

print(filtered_tokens)

['city', 'council', 'new', 'transport', 'plan', 'good']


**Compare the original text with the remaining tokens.**
If you wanted to study opinions about the transport plan, which word would you
keep rather than remove? Explain briefly.

I would keep “not”. The original says the plan is “not very good”, but the
remaining tokens contain “good” without the negation. That could lead us to
misinterpret the opinion. Keeping “very” may also help preserve the strength
of an opinion.

## 4. From texts to numbers

### a. Read a document-term matrix

**Run the next cell.** We use three short headlines to keep the table readable.
`CountVectorizer()` learns the vocabulary and counts its words in each headline.
The last lines display the counts as a labelled table.

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

headlines = [
    "Climate policy divides council",
    "Council debates climate policy",
    "Climate protests grow",
]

vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(headlines)

print("Matrix shape:", counts.shape)
display(pd.DataFrame(
    counts.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Headline 1", "Headline 2", "Headline 3"],
))

Matrix shape: (3, 7)


,climate,council,debates,divides,grow,policy,protests
Headline 1,1,1,0,1,0,1,0
Headline 2,1,1,1,0,0,1,0
Headline 3,1,0,0,0,1,0,1


**In the text cell below**, explain what the rows, columns, and numbers in the
table represent. What does the “climate” column tell you about these headlines?

**Rows:** the three headlines, one per row.

**Columns:** the seven distinct words in the vocabulary.

**Numbers:** the number of times a word occurs in a headline.

**The “climate” column:** “climate” occurs once in each headline. This does not
mean that the headlines say the same thing.

The matrix shape is `(3, 7)`. `toarray()` lets us inspect this tiny example;
for a large collection, keep the sparse matrix to save memory.

### b. Keep neighbouring words together

**Run the code below with `(1, 1)` first. Then change `ngram_range` to `(1, 3)`
and run it again.** This adds bigrams and trigrams alongside the single words.

In [9]:
phrases = ["very good", "not very good"]

phrase_vectorizer = CountVectorizer(ngram_range=(1, 3))
phrase_counts = phrase_vectorizer.fit_transform(phrases)

display(pd.DataFrame(
    phrase_counts.toarray(),
    columns=phrase_vectorizer.get_feature_names_out(),
    index=phrases,
))

,good,not,not very,not very good,very,very good
very good,1,0,0,0,1,1
not very good,1,1,1,1,1,1


**Name one new bigram and one new trigram from the table.**
Why might the trigram be useful when studying opinions?

**Bigram:** “not very” or “very good”.

**Trigram:** “not very good”.

**Why it may be useful:** the trigram keeps the negation together with the
description. The unigram counts do include “not”, but they do not preserve
which words follow it. N-grams preserve short phrases; they do not give a
model a full understanding of a text.

These examples show why preprocessing is a research choice: what you remove
and what you count can affect your conclusions. Next week we will build on
these text representations to explore unsupervised methods.

For reference: [CountVectorizer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

## Optional: stemming and lemmatisation

Finish Parts 1–4 first. For this extra example you need NLTK, spaCy, and spaCy's
English model. If you need them, uncomment and run the installation lines below.
You can skip this section if they are not installed.

In [10]:
# %pip install nltk spacy
# import sys
# !{sys.executable} -m spacy download en_core_web_sm

**Run both examples.** Stemming shortens word forms; the result need not be a
dictionary word. Lemmatisation aims to find a word's base form. We give spaCy a
complete sentence so it can use the surrounding words.

In [11]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
words = ["report", "reports", "reporting"]

for word in words:
    print(word, "->", stemmer.stem(word))

report -> report
reports -> report
reporting -> report


In [12]:
import spacy

nlp = spacy.load("en_core_web_sm")
sentence = "The reporters were writing about protests."

for token in nlp(sentence):
    print(token.text, "->", token.lemma_)

The -> the
reporters -> reporter
were -> be
writing -> write
about -> about
protests -> protest
. -> .


**Point out one change in each example.** Why might you want to count different
forms of a word together?

The stemmer turns “reports” and “reporting” into “report”. spaCy turns
“writing” into “write” and “protests” into “protest”.
Grouping word forms can help us count mentions of the same action or topic
together. Whether we want that depends on the research question.